In [1]:
%pip cache purge
%pip install -r ../requirements.txt


Files removed: 0 (0 bytes)
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import shutil

DIRECTORIES = [
    "../models", 
    "../data/raw/files"
]
deleted_files = []

for directory in DIRECTORIES:
    if not os.path.exists(directory):
        print(f"Directory '{directory}' does not exist.")
        continue

    for item in os.listdir(directory):
        item_path = os.path.join(directory, item)

        if item == ".gitkeep":
            continue  # Skip .gitkeep

        if os.path.isfile(item_path):
            os.remove(item_path)
            deleted_files.append(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
            deleted_files.append(item_path + "/")  

if deleted_files:
    print("Deleted files and directories:")
    for file in deleted_files:
        print(f" - {file}")
else:
    print("No files to delete.")


Deleted files and directories:
 - ../models/eeg_dataset_20250316_181552/
 - ../data/raw/files/MNE-eegbci-data/


In [3]:
# Establecer semilla aleatoria para reproducibilidad
import numpy as np
import random
RANDOM_SEED = None
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [5]:
# Importar funciones de nuestros scripts
from preprocessing import load_subjects_for_experiment, normalize_labels, save_metadata, save_processed_data, clean_memory
# from pipeline import compare_pipelines, hold_one_out_experiment, train_and_save_model, plot_subject_performances, plot_confusion_matrices, pipeline_comparison_chart
from predict import load_model, load_specific_subject, predict_eeg, visualize_predictions_over_time


In [6]:
# Configuración
NUM_SUBJECTS = 5  # Número de sujetos a cargar

print(f"Descargando y preprocesando datos EEG de {NUM_SUBJECTS} sujetos aleatorios...")

# Descargar y preprocesar, eligiendo un grupo de experimento aleatorio
eeg_data = load_subjects_for_experiment(
    num_subjects=NUM_SUBJECTS, 
    experiment_group=None,  # None para selección aleatoria
    random_seed=RANDOM_SEED
)

# Normalizar etiquetas para asegurar compatibilidad entre diferentes paradigmas
eeg_data = normalize_labels(eeg_data)

# Resumen de los datos cargados
print("\nResumen de los datos EEG cargados:")
for i, info in enumerate(eeg_data):
    print(f"Sujeto {i+1}: ID={info['subject']}, Tarea={info['task_type']}, Paradigma={info['paradigm']}")
    print(f"  Forma de datos: {info['X'].shape}, Clases: {list(info['event_id'].keys())}")
    print(f"  Distribución de clases: {info['class_counts']}")
    print()

# Guardar datos procesados con metadatos completos
saved_info = save_processed_data(eeg_data)
print(f"\nDatos guardados en: {saved_info['dataset_dir']}")

# Imprimir la estructura para diagnosticar
print("Estructura de saved_info['config']:")
for key in saved_info['config']:
    print(f"  - {key}")

# Usar las claves correctas para acceder a la información
n_samples = saved_info['config'].get('n_samples') or saved_info['config'].get('dataset_info', {}).get('n_samples')
experiment_group = saved_info['config'].get('experiment_group') or saved_info['config'].get('dataset_info', {}).get('experiment_group')

print(f"Total de muestras: {n_samples}")
print(f"Grupo de experimento: {experiment_group}")

# Limpiar memoria
light_data = clean_memory(eeg_data)
print("Memoria limpiada: se eliminaron los arrays grandes de X e y")


2025-03-16 18:13:50,410 - eeg_preprocessing - INFO - Cargando sujeto: 10, run: 6
2025-03-16 18:13:50,411 - eeg_preprocessing - INFO - Tipo de tarea: motor_imagery, paradigma: hands_feet


Descargando y preprocesando datos EEG de 5 sujetos aleatorios...
Grupo de experimento seleccionado: Motor imagery: hands vs feet
Ejecutando experimentos con runs: [6, 10, 14]
Tipo de tarea: motor_imagery, Paradigma: hands_feet

Sujetos seleccionados: [10, 85, 3, 88, 23]

Cargando EEG #1:
Sujeto: 10, Run: 6


Download complete in 06s (2.4 MB)
Extracting EDF parameters from /workspaces/42_vortex/data/raw/files/MNE-eegbci-data/files/eegmmidb/1.0.0/S010/S010R06.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...


2025-03-16 18:13:57,304 - eeg_preprocessing - INFO - Reduciendo frecuencia de muestreo a 128 Hz...
2025-03-16 18:13:57,391 - eeg_preprocessing - INFO - Aplicando filtro pasa banda (8-40 Hz)...


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 213 samples (1.664 s)



...

Applying ICA to Raw instance
    Transforming to ICA space (15 components)
    Zeroing out 1 ICA component
    Projecting back using 64 PCA components
Used Annotations descriptions: ['T0', 'T1', 'T2']


2025-03-16 18:15:51,583 - eeg_preprocessing - INFO - Mapeo de eventos: {'rest': 1, 'both_hands': 2, 'both_feet': 3}
2025-03-16 18:15:51,583 - eeg_preprocessing - INFO - Excluyendo eventos de descanso (REST)...


Not setting metadata
15 matching events found
Applying baseline correction (mode: mean)
Created an SSP operator (subspace dimension = 1)
1 projection items activated
Using data from preloaded Raw for 15 events and 513 original time points ...
0 bad epochs dropped


2025-03-16 18:15:51,612 - eeg_preprocessing - INFO - Creadas 15 épocas con 64 canales
2025-03-16 18:15:51,614 - eeg_preprocessing - INFO - Extrayendo características mediante descomposición wavelet...
2025-03-16 18:15:51,614 - eeg_preprocessing - INFO - Aplicando transformada wavelet (db4, nivel 5)...
2025-03-16 18:15:52,801 - eeg_preprocessing - INFO - Características extraídas: X shape (15, 1920), y shape (15,)
2025-03-16 18:15:52,804 - eeg_preprocessing - INFO - Directorio ../models/eeg_dataset_20250316_181552 creado
2025-03-16 18:15:52,807 - eeg_preprocessing - INFO - Guardando arrays con formas: X=(237, 1920), y=(237,), subjects=(237,), runs=(237,)
2025-03-16 18:15:52,810 - eeg_preprocessing - INFO - Arrays guardados correctamente
2025-03-16 18:15:52,811 - eeg_preprocessing - INFO - Metadatos guardados en ../models/eeg_dataset_20250316_181552/dataset_info.json
2025-03-16 18:15:52,812 - eeg_preprocessing - INFO - Datos y metadatos completos guardados en ../models/eeg_dataset_202503

  Sujeto: 23, Run: 14, Tarea: motor_imagery, Paradigma: hands_feet
  Forma de datos: (15, 1920)
  Clases: ['both_hands', 'both_feet']
  Distribución de clases: {'both_hands': 8, 'both_feet': 7}

Cargados 15 EEGs exitosamente de 5 sujetos.

Resumen de los datos EEG cargados:
Sujeto 1: ID=10, Tarea=motor_imagery, Paradigma=hands_feet
  Forma de datos: (15, 1920), Clases: ['rest', 'clase1', 'clase2']
  Distribución de clases: {'rest': 0, 'clase1': 7, 'clase2': 8}

Sujeto 2: ID=10, Tarea=motor_imagery, Paradigma=hands_feet
  Forma de datos: (15, 1920), Clases: ['rest', 'clase1', 'clase2']
  Distribución de clases: {'rest': 0, 'clase1': 8, 'clase2': 7}

Sujeto 3: ID=10, Tarea=motor_imagery, Paradigma=hands_feet
  Forma de datos: (15, 1920), Clases: ['rest', 'clase1', 'clase2']
  Distribución de clases: {'rest': 0, 'clase1': 8, 'clase2': 7}

Sujeto 4: ID=85, Tarea=motor_imagery, Paradigma=hands_feet
  Forma de datos: (15, 1920), Clases: ['rest', 'clase1', 'clase2']
  Distribución de clases: 

In [ ]:
# Importar las librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from datetime import datetime

# Importar funciones desde los módulos
from preprocessing import load_subjects_for_experiment, normalize_labels, save_processed_data, clean_memory
from pipeline import compare_binary_pipelines, binary_hold_one_out_experiment, train_and_save_binary_model
from pipeline import plot_binary_subject_performances, plot_binary_confusion_matrices, binary_pipeline_comparison_chart
from pipeline import filter_rest_events

# Cargar el dataset desde archivos guardados
def load_dataset_from_files(dataset_dir):
    """
    Carga un dataset guardado previamente
    
    Args:
        dataset_dir (str): Ruta al directorio del dataset
        
    Returns:
        dict: Datos cargados
    """
    print(f"Cargando dataset desde: {dataset_dir}")
    
    # Cargar arrays
    X_all = np.load(os.path.join(dataset_dir, 'X_all.npy'))
    y_all = np.load(os.path.join(dataset_dir, 'y_all.npy'))
    subjects_all = np.load(os.path.join(dataset_dir, 'subjects_all.npy'))
    runs_all = np.load(os.path.join(dataset_dir, 'runs_all.npy'))
    
    # Cargar configuración
    with open(os.path.join(dataset_dir, 'dataset_info.json'), 'r') as f:
        config = json.load(f)
    
    # Reconstruir datos en formato de lista de diccionarios para compatibilidad
    eeg_data = []
    
    for subject_info in config['subjects_data']:
        subject_id = subject_info['subject']
        run_id = subject_info['run']
        
        # Identificar índices para este sujeto/run
        start_idx = subject_info['sample_indices']['start']
        end_idx = subject_info['sample_indices']['end']
        
        # Extraer datos para este sujeto
        X_subject = X_all[start_idx:end_idx]
        y_subject = y_all[start_idx:end_idx]
        
        # Crear diccionario con información
        subject_data = {
            'subject': subject_id,
            'run': run_id,
            'task_type': subject_info['task_type'],
            'paradigm': subject_info['paradigm'],
            'experiment_group': subject_info['experiment_group'],
            'X': X_subject,
            'y': y_subject,
            'event_id': {'rest': 1, 'clase1': 2, 'clase2': 3},
            'class_counts': subject_info['class_counts']
        }
        
        eeg_data.append(subject_data)
    
    print(f"Dataset cargado exitosamente: {len(eeg_data)} registros EEG")
    print(f"Grupo de experimento: {config['dataset_info']['experiment_group']}")
    print(f"Total de muestras: {X_all.shape[0]}, Características: {X_all.shape[1]}")
    
    return {
        'eeg_data': eeg_data,
        'config': config,
        'X_all': X_all,
        'y_all': y_all,
        'subjects_all': subjects_all,
        'runs_all': runs_all
    }

# Paso 1: Encontrar el directorio del dataset más reciente
def find_latest_dataset_dir(base_dir='../models'):
    """Encuentra el directorio de dataset más reciente"""
    dataset_dirs = [d for d in os.listdir(base_dir) if d.startswith('eeg_dataset_')]
    if not dataset_dirs:
        raise ValueError(f"No se encontraron datasets en {base_dir}")
    
    # Ordenar por timestamp (parte del nombre)
    latest_dir = sorted(dataset_dirs)[-1]
    return os.path.join(base_dir, latest_dir)

# Paso 2: Cargar el dataset
latest_dataset_dir = find_latest_dataset_dir()
print(f"Usando el dataset más reciente: {latest_dataset_dir}")

dataset = load_dataset_from_files(latest_dataset_dir)
eeg_data = dataset['eeg_data']

# Filtrar eventos de reposo (rest)
print("\n=== Filtrando eventos 'rest' para clasificación binaria ===")
binary_eeg_data = filter_rest_events(eeg_data)

# Paso 3: Comparar diferentes pipelines de clasificación binaria
print("\n=== Comparación de pipelines para clasificación binaria ===")
pipeline_configs = ['csp_svm', 'csp_freq_rf', 'asymmetry_rf', 'pca_mlp', 'ensemble', 'enhanced_csp_rf']
binary_comparison_results = compare_binary_pipelines(binary_eeg_data, configs=pipeline_configs)

# Paso 4: Visualizar comparación de pipelines binarios
fig = binary_pipeline_comparison_chart(binary_comparison_results)
plt.savefig(os.path.join('../models', 'binary_pipeline_comparison.png'), dpi=300, bbox_inches='tight')
plt.close(fig)

# Paso 5: Evaluar el mejor pipeline con hold-one-out cross-validation
best_binary_pipeline = binary_comparison_results['best_config']
print(f"\n=== Evaluación del mejor pipeline ({best_binary_pipeline}) con hold-one-out ===")
binary_holdout_results = binary_hold_one_out_experiment(binary_eeg_data, pipeline_config=best_binary_pipeline)

# Paso 6: Visualizar resultados por sujeto
fig_perf = plot_binary_subject_performances(binary_holdout_results)
plt.savefig(os.path.join('../models', 'binary_subject_performances.png'), dpi=300, bbox_inches='tight')
plt.close(fig_perf)

fig_cm = plot_binary_confusion_matrices(binary_holdout_results)
plt.savefig(os.path.join('../models', 'binary_confusion_matrices.png'), dpi=300, bbox_inches='tight')
plt.close(fig_cm)

# Paso 7: Entrenar y guardar el modelo final binario
print("\n=== Entrenamiento del modelo final para clasificación binaria ===")
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_name = f'eeg_binary_model_{dataset["config"]["dataset_info"]["experiment_group"]}_{timestamp}'
final_model, model_info = train_and_save_binary_model(binary_eeg_data, pipeline_config=best_binary_pipeline, model_name=model_name)

print("\n=== Resumen de la evaluación para clasificación binaria ===")
print(f"Mejor pipeline: {best_binary_pipeline}")
print(f"Accuracy CV: {binary_comparison_results['results'][best_binary_pipeline]['accuracy']:.4f} ± {binary_comparison_results['results'][best_binary_pipeline]['accuracy_std']:.4f}")
print(f"Hold-one-out Accuracy: {binary_holdout_results['avg_accuracy']:.4f}")
print(f"Modelo final guardado como: {model_info['model_file']}")
print(f"Gráficos de resultados guardados en el directorio '../models'")


In [ ]:
from predict_with_same_experiment import predict_with_same_experiment

import os
import glob

# Encontrar el modelo .joblib más reciente
model_files = glob.glob('../models/*.joblib')
if model_files:
    model_files.sort(key=lambda x: os.path.getmtime(x), reverse=True)
    latest_model = model_files[0]
    print(f"Usando el modelo más reciente: {latest_model}")
    
    # Ahora puedes pasar este modelo específico
    prediction_results = predict_with_same_experiment(num_subjects=6, model_path=latest_model)
else:
    print("No se encontraron modelos .joblib en el directorio ../models/")
